In [2]:
# =================================================================================================
# NHANES 2009-2010 DIETARY MERGE - FIDELITY PRESERVING
#
# Source files:
#
#   DR1TOT_F_corrected.csv
#   DR2TOT_F_corrected.csv
#
# Purpose:
#
#   Merge NHANES Dietary Day 1 + Day 2 Total Nutrient Intake files
#   into one participant-level Dietary CSV.
#
#
# Final structure:
#
#   SEQN
#
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#
#   [Day 1 variables]
#
#   [Day 2 variables]
#
#
# Fidelity rules:
#
#   - One SEQN per participant
#
#   - Preserve already-corrected Dietary decimal text AS IS
#
#   - Do NOT round
#
#   - Do NOT use float_format=
#
#   - Do NOT integerize Dietary nutrient variables
#
#   - Validate shared Day 1 / Day 2 variables before removing
#     the redundant Day 2 copies
#
#   - Prevent engineered duplicate fields:
#
#       WTDRD1_DAY_2_TOTAL_INTAKE
#       WTDR2D_DAY_2_TOTAL_INTAKE
#       DRABF_DAY_2_TOTAL_INTAKE
#       DRDINT_DAY_2_TOTAL_INTAKE
#
#   - Every participant-level merge is one-to-one
#
#   - Validate source-to-merged text fidelity
#
#   - Validate pre-save/post-save text fidelity
#
#
# CDC 2009-2010 benchmarks:
#
#   DR1TOT_F rows        = 10,253
#   DR2TOT_F rows        = 10,253
#
#   WTDRD1 populated     = 9,754
#   WTDR2D populated     = 8,406
#
#   DRDINT = 1           = 1,348
#   DRDINT = 2           = 8,406
#   DRDINT missing       =   499
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Dietary"
)


DAY1_FILE = (
    DATA_DIR
    / "DR1TOT_F_corrected.csv"
)


DAY2_FILE = (
    DATA_DIR
    / "DR2TOT_F_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2009_2010"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2009_2010_raw_corrected.csv"
)


# =================================================================================================
# 3. READ CORRECTED CSV FILES AS TEXT
# =================================================================================================

#
# VERY IMPORTANT:
#
# dtype=str prevents pandas from re-parsing and potentially
# re-formatting already-corrected Dietary decimal values.
#
#
# Example:
#
#     0.028999999999
#
# remains:
#
#     0.028999999999
#
#
# This notebook is performing a MERGE.
#
# It is NOT recalculating nutrient values.
#

day1 = pd.read_csv(
    DAY1_FILE,
    dtype=str,
    keep_default_na=False
)


day2 = pd.read_csv(
    DAY2_FILE,
    dtype=str,
    keep_default_na=False
)


print("=" * 110)

print(
    "NHANES 2009-2010 DIETARY DAY 1 + DAY 2 MERGE"
)

print("=" * 110)


print(
    "DR1TOT_F shape:",
    day1.shape
)


print(
    "DR2TOT_F shape:",
    day2.shape
)


# =================================================================================================
# 4. STANDARDIZE COLUMN NAMES
# =================================================================================================

day1.columns = (
    day1.columns
    .str.strip()
    .str.upper()
)


day2.columns = (
    day2.columns
    .str.strip()
    .str.upper()
)


# =================================================================================================
# 5. CYCLE-SPECIFIC EXPECTED COUNTS
# =================================================================================================

EXPECTED_ROWS = 10253


EXPECTED_DAY1_WEIGHT_NONMISSING = 9754


EXPECTED_DAY2_WEIGHT_NONMISSING = 8406


EXPECTED_DRDINT = {

    "1": 1348,

    "2": 8406,

    "": 499

}


# =================================================================================================
# 6. VERIFY SEQN EXISTS
# =================================================================================================

for name, dataframe in [

    ("DR1TOT_F", day1),

    ("DR2TOT_F", day2)

]:

    if "SEQN" not in dataframe.columns:

        raise ValueError(
            f"{name} does not contain SEQN."
        )


print(
    "PASS: SEQN exists in both files."
)


# =================================================================================================
# 7. VERIFY OFFICIAL ROW COUNTS
# =================================================================================================

if len(day1) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR1TOT_F row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day1):,}"
    )


if len(day2) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR2TOT_F row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day2):,}"
    )


print(
    "PASS: Both DR1TOT_F and DR2TOT_F "
    "contain 10,253 participant rows."
)


# =================================================================================================
# 8. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataframe,
    name
):

    print(
        f"\n--- SEQN CHECK: {name} ---"
    )


    blank_count = int(

        dataframe["SEQN"]
        .eq("")
        .sum()

    )


    duplicate_count = int(

        dataframe["SEQN"]
        .duplicated()
        .sum()

    )


    dot_zero_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            dataframe["SEQN"]
            .str.endswith(".0")
        )

        .sum()

    )


    invalid_format_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            ~dataframe["SEQN"]
            .str.fullmatch(r"\d+")
        )

        .sum()

    )


    print(
        "Rows:",
        f"{len(dataframe):,}"
    )


    print(
        "Unique SEQN:",
        f"{dataframe['SEQN'].nunique():,}"
    )


    print(
        "Blank SEQN:",
        blank_count
    )


    print(
        "Duplicate SEQN:",
        duplicate_count
    )


    print(
        "SEQN ending '.0':",
        dot_zero_count
    )


    print(
        "Invalid SEQN formatting:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{name}: SEQN still contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{name}: invalid SEQN formatting."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 9. VALIDATE SEQN IN BOTH FILES
# =================================================================================================

validate_seqn(
    day1,
    "DR1TOT_F"
)


validate_seqn(
    day2,
    "DR2TOT_F"
)


print(
    "\nPASS: SEQN integrity validated in both files."
)


# =================================================================================================
# 10. VERIFY SAME PARTICIPANT UNIVERSE
# =================================================================================================

day1_seqn = set(
    day1["SEQN"]
)


day2_seqn = set(
    day2["SEQN"]
)


only_day1 = (
    day1_seqn
    -
    day2_seqn
)


only_day2 = (
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- PARTICIPANT SET CHECK ---"
)


print(
    "Only in DR1TOT_F:",
    len(only_day1)
)


print(
    "Only in DR2TOT_F:",
    len(only_day2)
)


if only_day1 or only_day2:

    print(
        "\nExample SEQN only in Day 1:"
    )


    print(
        sorted(
            only_day1
        )[:20]
    )


    print(
        "\nExample SEQN only in Day 2:"
    )


    print(
        sorted(
            only_day2
        )[:20]
    )


    raise ValueError(
        "DR1TOT_F and DR2TOT_F do not contain "
        "the same participant universe."
    )


print(
    "PASS: Both files contain the same "
    "10,253 participant SEQN."
)


# =================================================================================================
# 11. IDENTIFY SHARED COLUMN NAMES
# =================================================================================================

shared_columns_detected = [

    col

    for col in day1.columns

    if (
        col in day2.columns
        and
        col != "SEQN"
    )

]


print(
    "\n--- SHARED VARIABLES DETECTED ---"
)


print(
    shared_columns_detected
)


# =================================================================================================
# 12. EXPECTED SHARED NHANES VARIABLES
# =================================================================================================

#
# These official NHANES variables occur in both total-intake files.
#
# They describe shared dietary/sample information rather than
# separate nutrient measurements unique to Day 1 or Day 2.
#

SHARED_COLUMNS = [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


if set(
    shared_columns_detected
) != set(
    SHARED_COLUMNS
):

    raise ValueError(
        "Unexpected shared-column pattern.\n\n"
        f"Expected:\n{SHARED_COLUMNS}\n\n"
        f"Detected:\n{shared_columns_detected}"
    )


print(
    "PASS: Exactly four expected shared variables found."
)


# =================================================================================================
# 13. COMPARE SHARED VARIABLES BY SEQN
# =================================================================================================

#
# IMPORTANT:
#
# Files were deliberately loaded as strings.
#
# Therefore:
#
#     "8"
#
# and:
#
#     "8.0"
#
# are textually different.
#
#
# For cross-source validation we compare NUMERIC COPIES.
#
# The original text is NOT modified.
#

shared_comparison = (

    day1[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="outer",

        suffixes=(
            "_DAY1",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


print(
    "\n--- SHARED VARIABLE FIDELITY CHECK ---"
)


for col in SHARED_COLUMNS:

    left_raw = (
        shared_comparison[
            f"{col}_DAY1"
        ]
    )


    right_raw = (
        shared_comparison[
            f"{col}_DAY2"
        ]
    )


    left_missing = (
        left_raw.eq("")
    )


    right_missing = (
        right_raw.eq("")
    )


    missing_mismatch_count = int(

        (
            left_missing
            !=
            right_missing
        )

        .sum()

    )


    both_populated = (
        ~left_missing
        &
        ~right_missing
    )


    left_numeric = pd.to_numeric(

        left_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    right_numeric = pd.to_numeric(

        right_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        left_numeric.to_numpy(
            dtype=float
        ),

        right_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(

        (
            ~numeric_equal
        )

        .sum()

    )


    left_text = (

        left_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    right_text = (

        right_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    text_difference_count = int(

        (
            left_text
            !=
            right_text
        )

        .sum()

    )


    print(
        f"\n{col}"
    )


    print(
        "  Both populated:",
        f"{int(both_populated.sum()):,}"
    )


    print(
        "  Missing-position mismatches:",
        f"{missing_mismatch_count:,}"
    )


    print(
        "  True numeric mismatches:",
        f"{numeric_mismatch_count:,}"
    )


    print(
        "  Text-format differences:",
        f"{text_difference_count:,}"
    )


    if missing_mismatch_count != 0:

        raise ValueError(
            f"{col}: missing positions differ "
            "between DR1TOT_F and DR2TOT_F."
        )


    if numeric_mismatch_count != 0:

        mismatch_rows = (

            shared_comparison.loc[
                both_populated
            ]

            .reset_index(
                drop=True
            )

        )


        mismatch_rows[
            "NUMERIC_MATCH"
        ] = numeric_equal


        print(
            "\nTRUE NUMERIC MISMATCH EXAMPLES:"
        )


        display(

            mismatch_rows.loc[
                ~mismatch_rows[
                    "NUMERIC_MATCH"
                ],
                [
                    "SEQN",
                    f"{col}_DAY1",
                    f"{col}_DAY2"
                ]
            ]

            .head(20)

        )


        raise ValueError(
            f"{col}: true numeric values differ "
            "between DR1TOT_F and DR2TOT_F."
        )


print(
    "\nPASS: WTDRD1, WTDR2D, DRABF and DRDINT "
    "agree between Day 1 and Day 2."
)


# =================================================================================================
# 14. CDC WTDRD1 / WTDR2D SAMPLE-SIZE CHECK
# =================================================================================================

day1_weight_count = int(

    day1["WTDRD1"]
    .ne("")
    .sum()

)


day2_weight_count = int(

    day1["WTDR2D"]
    .ne("")
    .sum()

)


print(
    "\n--- CDC DIETARY WEIGHT CHECK ---"
)


print(
    "WTDRD1 populated:",
    f"{day1_weight_count:,}"
)


print(
    "WTDR2D populated:",
    f"{day2_weight_count:,}"
)


if (
    day1_weight_count
    !=
    EXPECTED_DAY1_WEIGHT_NONMISSING
):

    raise ValueError(
        "WTDRD1 count differs from CDC.\n"
        f"Expected: {EXPECTED_DAY1_WEIGHT_NONMISSING:,}\n"
        f"Actual:   {day1_weight_count:,}"
    )


if (
    day2_weight_count
    !=
    EXPECTED_DAY2_WEIGHT_NONMISSING
):

    raise ValueError(
        "WTDR2D count differs from CDC.\n"
        f"Expected: {EXPECTED_DAY2_WEIGHT_NONMISSING:,}\n"
        f"Actual:   {day2_weight_count:,}"
    )


print(
    "PASS: Dietary weight counts match CDC:"
)


print(
    "  WTDRD1 = 9,754"
)


print(
    "  WTDR2D = 8,406"
)


# =================================================================================================
# 15. CDC DRDINT CHECK
# =================================================================================================

drdint_counts = (

    day1["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRDINT CHECK ---"
)


print(
    drdint_counts
)


if drdint_counts != EXPECTED_DRDINT:

    raise ValueError(
        "DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{drdint_counts}"
    )


print(
    "PASS: DRDINT matches CDC."
)


# =================================================================================================
# 16. CDC DRABF CHECK
# =================================================================================================

#
# CDC 2009-2010:
#
# 1 = Yes      136
# 2 = No     9,618
# Missing      499
#

EXPECTED_DRABF = {

    "1": 136,

    "2": 9618,

    "": 499

}


drabf_counts = (

    day1["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRABF CHECK ---"
)


print(
    drabf_counts
)


if drabf_counts != EXPECTED_DRABF:

    raise ValueError(
        "DRABF distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{drabf_counts}"
    )


print(
    "PASS: DRABF matches CDC."
)


# =================================================================================================
# 17. SAVE ORIGINAL SOURCE COLUMN ORDER
# =================================================================================================

original_day1_columns = (
    day1.columns.tolist()
)


original_day2_columns = (
    day2.columns.tolist()
)


# =================================================================================================
# 18. REMOVE ONLY REDUNDANT DAY 2 SHARED COPIES
# =================================================================================================

#
# Keep canonical copies from DR1TOT_F.
#
# Remove identical copies from DR2TOT_F BEFORE merging.
#
#
# This prevents pandas from creating:
#
#     WTDRD1_DAY_2_TOTAL_INTAKE
#     WTDR2D_DAY_2_TOTAL_INTAKE
#     DRABF_DAY_2_TOTAL_INTAKE
#     DRDINT_DAY_2_TOTAL_INTAKE
#

day2_for_merge = (

    day2.drop(
        columns=SHARED_COLUMNS
    )

)


print(
    "\nRemoved redundant DR2TOT_F copies:"
)


for col in SHARED_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 19. VERIFY NO OTHER COLUMN COLLISIONS REMAIN
# =================================================================================================

remaining_overlap = sorted(

    (
        set(
            day1.columns
        )

        &

        set(
            day2_for_merge.columns
        )
    )

    -

    {
        "SEQN"
    }

)


if remaining_overlap:

    raise ValueError(
        "Unexpected overlapping columns remain before merge:\n"
        + str(
            remaining_overlap
        )
    )


print(
    "PASS: No unexpected column collisions remain."
)


# =================================================================================================
# 20. MERGE DAY 1 + DAY 2
# =================================================================================================

dietary = (

    day1.merge(

        day2_for_merge,

        on="SEQN",

        how="outer",

        validate="one_to_one"

    )

)


#
# Outer joins can create NaN where a row was absent
# from a particular source.
#
# Convert those merge-generated NaNs back to CSV blanks.
#

dietary = (
    dietary.fillna("")
)


print(
    "\nPASS: DR1TOT_F + DR2TOT_F merge completed."
)


# =================================================================================================
# 21. FINAL PARTICIPANT COUNT CHECK
# =================================================================================================

if len(dietary) != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary row count changed.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(dietary):,}"
    )


if (
    dietary["SEQN"]
    .nunique()
    !=
    EXPECTED_ROWS
):

    raise ValueError(
        "Merged Dietary unique SEQN count changed."
    )


if (
    dietary["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Merged Dietary dataset contains duplicate SEQN."
    )


if (
    dietary["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Merged Dietary dataset contains blank SEQN."
    )


if (
    dietary["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Merged Dietary SEQN contains '.0'."
    )


print(
    "PASS: Final Dietary dataset contains "
    "10,253 unique participant SEQN."
)


# =================================================================================================
# 22. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_column_names = (

    dietary.columns[
        dietary.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_column_names
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 23. ENGINEERED DUPLICATE COLUMN CHECK
# =================================================================================================

BAD_DUPLICATE_COLUMNS = [

    "WTDRD1_DAY_2_TOTAL_INTAKE",

    "WTDR2D_DAY_2_TOTAL_INTAKE",

    "DRABF_DAY_2_TOTAL_INTAKE",

    "DRDINT_DAY_2_TOTAL_INTAKE",

    "WTDRD1_X",

    "WTDRD1_Y",

    "WTDR2D_X",

    "WTDR2D_Y",

    "DRABF_X",

    "DRABF_Y",

    "DRDINT_X",

    "DRDINT_Y"

]


existing_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in dietary.columns

]


if existing_bad_columns:

    raise ValueError(
        "Engineered Dietary duplicate columns remain:\n"
        + str(
            existing_bad_columns
        )
    )


print(
    "PASS: No engineered Day 2 duplicate columns."
)


# =================================================================================================
# 24. CREATE FINAL LOGICAL COLUMN ORDER
# =================================================================================================

#
# Desired:
#
#   SEQN
#
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#
#   [Day 1 unique fields]
#
#   [Day 2 unique fields]
#

SHARED_ORDER = [

    "SEQN",

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


DAY1_UNIQUE_COLUMNS = [

    col

    for col in original_day1_columns

    if col not in SHARED_ORDER

]


DAY2_UNIQUE_COLUMNS = [

    col

    for col in original_day2_columns

    if (
        col != "SEQN"
        and
        col not in SHARED_COLUMNS
    )

]


FINAL_COLUMN_ORDER = (

    SHARED_ORDER

    +

    DAY1_UNIQUE_COLUMNS

    +

    DAY2_UNIQUE_COLUMNS

)


# =================================================================================================
# 25. VALIDATE FINAL COLUMN ORDER
# =================================================================================================

if (
    len(FINAL_COLUMN_ORDER)
    !=
    len(set(FINAL_COLUMN_ORDER))
):

    raise ValueError(
        "FINAL_COLUMN_ORDER contains duplicate names."
    )


missing_from_order = (

    set(
        dietary.columns
    )

    -

    set(
        FINAL_COLUMN_ORDER
    )

)


extra_in_order = (

    set(
        FINAL_COLUMN_ORDER
    )

    -

    set(
        dietary.columns
    )

)


if missing_from_order or extra_in_order:

    raise ValueError(
        "Final column-order mismatch.\n\n"
        f"Missing from order:\n"
        f"{sorted(missing_from_order)}\n\n"
        f"Unexpected in order:\n"
        f"{sorted(extra_in_order)}"
    )


dietary = (
    dietary[
        FINAL_COLUMN_ORDER
    ]
)


print(
    "PASS: Final column order created."
)


# =================================================================================================
# 26. DISPLAY FINAL STRUCTURE
# =================================================================================================

print(
    "\n--- FINAL DIETARY STRUCTURE ---"
)


print(
    "Shared fields:",
    len(SHARED_ORDER)
)


print(
    "Day 1 unique fields:",
    len(DAY1_UNIQUE_COLUMNS)
)


print(
    "Day 2 unique fields:",
    len(DAY2_UNIQUE_COLUMNS)
)


print(
    "Total fields:",
    len(dietary.columns)
)


# =================================================================================================
# 27. DAY 1 SOURCE-TO-MERGED TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DAY 1 SOURCE FIDELITY ---"
)


day1_check = (

    day1.merge(

        dietary[
            original_day1_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day1_failures = []


for col in original_day1_columns:

    if col == "SEQN":

        continue


    source_values = (
        day1_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day1_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day1_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day1_failures:

    raise ValueError(
        "DR1TOT_F values changed during merge:\n"
        + str(
            day1_failures
        )
    )


print(
    "PASS: Every retained DR1TOT_F value "
    "preserved text-for-text."
)


# =================================================================================================
# 28. DAY 2 UNIQUE FIELD SOURCE FIDELITY
# =================================================================================================

print(
    "\n--- DAY 2 SOURCE FIDELITY ---"
)


day2_check_columns = (

    ["SEQN"]

    +

    DAY2_UNIQUE_COLUMNS

)


day2_check = (

    day2[
        day2_check_columns
    ]

    .merge(

        dietary[
            day2_check_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day2_failures = []


for col in DAY2_UNIQUE_COLUMNS:

    source_values = (
        day2_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day2_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day2_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day2_failures:

    raise ValueError(
        "DR2TOT_F values changed during merge:\n"
        + str(
            day2_failures
        )
    )


print(
    "PASS: Every unique DR2TOT_F value "
    "preserved text-for-text."
)


# =================================================================================================
# 29. FINAL SHARED FIELD CHECK AGAINST DAY 2
# =================================================================================================

print(
    "\n--- FINAL SHARED FIELD CHECK ---"
)


final_shared_check = (

    dietary[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="inner",

        suffixes=(
            "_FINAL",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


for col in SHARED_COLUMNS:

    final_raw = (
        final_shared_check[
            f"{col}_FINAL"
        ]
    )


    day2_raw = (
        final_shared_check[
            f"{col}_DAY2"
        ]
    )


    final_missing = (
        final_raw.eq("")
    )


    day2_missing = (
        day2_raw.eq("")
    )


    missing_mismatch = int(

        (
            final_missing
            !=
            day2_missing
        )

        .sum()

    )


    both_populated = (
        ~final_missing
        &
        ~day2_missing
    )


    final_numeric = pd.to_numeric(

        final_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    day2_numeric = pd.to_numeric(

        day2_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_mismatch = int(

        (
            ~np.isclose(

                final_numeric.to_numpy(
                    dtype=float
                ),

                day2_numeric.to_numpy(
                    dtype=float
                ),

                rtol=0,

                atol=0

            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"missing mismatches={missing_mismatch:,}, "
        f"numeric mismatches={numeric_mismatch:,}"
    )


    if missing_mismatch != 0:

        raise ValueError(
            f"{col}: final missing positions differ "
            "from DR2TOT_F."
        )


    if numeric_mismatch != 0:

        raise ValueError(
            f"{col}: final value differs "
            "from DR2TOT_F."
        )


print(
    "PASS: Shared fields remain equivalent "
    "to both source files."
)


# =================================================================================================
# 30. DIETARY DECIMAL PROTECTION
# =================================================================================================

#
# These checks are diagnostic.
#
# NO conversion or rounding is performed.
#

REPRESENTATIVE_DECIMAL_FIELDS = [

    "WTDRD1",

    "WTDR2D",

    "DR1TPROT",

    "DR1TCARB",

    "DR1TP204",

    "DR1_320Z",

    "DR2TPROT",

    "DR2TCARB",

    "DR2TP204",

    "DR2_320Z"

]


print(
    "\n--- DIETARY DECIMAL PRESERVATION ---"
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in dietary.columns:

        continue


    populated_count = int(

        dietary[col]
        .ne("")
        .sum()

    )


    decimal_token_count = int(

        (
            dietary[col].ne("")
            &
            dietary[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: Dietary values remained text; "
    "no merge-time rounding or integer conversion."
)


# =================================================================================================
# 31. KNOWN XPORT TINY-ARTIFACT TOKEN CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_found = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]
        .str.lower()
        .eq(
            TINY_TOKEN.lower()
        )
        .sum()

    )


    if count > 0:

        tiny_found[
            col
        ] = count


print(
    "\nKnown tiny XPORT artifacts:",
    sum(
        tiny_found.values()
    )
)


if tiny_found:

    raise ValueError(
        "Known XPORT tiny-zero artifacts remain "
        "in corrected Dietary input:\n"
        + str(
            tiny_found
        )
    )


print(
    "PASS: No known tiny XPORT artifacts."
)


# =================================================================================================
# 32. SCIENTIFIC NOTATION TOKEN CHECK
# =================================================================================================

scientific_notation_fields = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()

    )


    if count > 0:

        scientific_notation_fields[
            col
        ] = count


if scientific_notation_fields:

    raise ValueError(
        "Unexpected scientific notation tokens found:\n"
        + str(
            scientific_notation_fields
        )
    )


print(
    "PASS: No scientific notation tokens."
)


# =================================================================================================
# 33. SAVE FINAL CSV
# =================================================================================================

#
# IMPORTANT:
#
# No float_format=
#
# No numeric conversion
#
# No rounding
#
# The corrected Dietary text is written AS IS.
#

dietary.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved final Dietary file:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 34. RE-READ FINAL CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 35. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != dietary.shape:

    raise ValueError(
        "Saved CSV shape changed.\n"
        f"Before: {dietary.shape}\n"
        f"After:  {final_df.shape}"
    )


if (
    final_df.columns.tolist()
    !=
    dietary.columns.tolist()
):

    raise ValueError(
        "Saved CSV column structure changed."
    )


print(
    "PASS: Saved CSV structure preserved."
)


# =================================================================================================
# 36. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

post_save_failures = []


for col in dietary.columns:

    same = (

        dietary[col]
        .eq(
            final_df[col]
        )
        .all()

    )


    if not same:

        post_save_failures.append(
            col
        )


if post_save_failures:

    raise ValueError(
        "Text changed during CSV export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 37. FINAL SEQN VALIDATION
# =================================================================================================

if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (
    ~final_df["SEQN"]
    .str.fullmatch(r"\d+")
).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 38. FINAL ENGINEERED-DUPLICATE CHECK
# =================================================================================================

final_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in final_df.columns

]


if final_bad_columns:

    raise ValueError(
        "Engineered duplicate columns remain:\n"
        + str(
            final_bad_columns
        )
    )


print(
    "PASS: No redundant Day 2 shared-variable copies."
)


# =================================================================================================
# 39. FINAL CDC SAMPLE-SIZE CHECK
# =================================================================================================

final_wtdrd1_count = int(

    final_df["WTDRD1"]
    .ne("")
    .sum()

)


final_wtdr2d_count = int(

    final_df["WTDR2D"]
    .ne("")
    .sum()

)


if final_wtdrd1_count != 9754:

    raise ValueError(
        "Final WTDRD1 count differs from CDC.\n"
        f"Expected: 9,754\n"
        f"Actual:   {final_wtdrd1_count:,}"
    )


if final_wtdr2d_count != 8406:

    raise ValueError(
        "Final WTDR2D count differs from CDC.\n"
        f"Expected: 8,406\n"
        f"Actual:   {final_wtdr2d_count:,}"
    )


print(
    "PASS: Final Dietary weight counts match CDC."
)


# =================================================================================================
# 40. FINAL DRDINT CHECK
# =================================================================================================

final_drdint_counts = (

    final_df["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


if final_drdint_counts != EXPECTED_DRDINT:

    raise ValueError(
        "Final DRDINT distribution differs.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{final_drdint_counts}"
    )


print(
    "PASS: Final DRDINT distribution matches CDC."
)


# =================================================================================================
# 41. FINAL TINY ARTIFACT CHECK AFTER SAVE
# =================================================================================================

final_tiny_found = {}


for col in final_df.columns:

    if col == "SEQN":

        continue


    count = int(

        final_df[col]
        .str.lower()
        .eq(
            TINY_TOKEN.lower()
        )
        .sum()

    )


    if count > 0:

        final_tiny_found[
            col
        ] = count


if final_tiny_found:

    raise ValueError(
        "Tiny XPORT artifacts remain after CSV save:\n"
        + str(
            final_tiny_found
        )
    )


print(
    "PASS: No tiny XPORT artifacts after CSV save."
)


# =================================================================================================
# 42. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2009-2010 DIETARY MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2009-2010"
)


print(
    ""
)


print(
    "Source files:"
)


print(
    "  DR1TOT_F_corrected.csv"
)


print(
    "  DR2TOT_F_corrected.csv"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    ""
)


print(
    "CDC Dietary benchmarks:"
)


print(
    "  Total participant rows = 10,253"
)


print(
    "  WTDRD1 populated        = 9,754"
)


print(
    "  WTDR2D populated        = 8,406"
)


print(
    ""
)


print(
    "DRDINT:"
)


print(
    "  Day 1 only       = 1,348"
)


print(
    "  Day 1 + Day 2    = 8,406"
)


print(
    "  Missing          =   499"
)


print(
    ""
)


print(
    "Shared fields retained once:"
)


print(
    "  WTDRD1"
)


print(
    "  WTDR2D"
)


print(
    "  DRABF"
)


print(
    "  DRDINT"
)


print(
    ""
)


print(
    "Final organization:"
)


print(
    "  SEQN"
)


print(
    "  Shared Dietary fields"
)


print(
    "  Day 1 fields"
)


print(
    "  Day 2 fields"
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Engineered Dietary duplicates: 0"
)


print(
    "Known tiny XPORT artifacts: 0"
)


print(
    "Day 1 retained-value changes: 0"
)


print(
    "Day 2 retained-value changes: 0"
)


print(
    "Dietary decimal reformatting during merge: 0"
)


print(
    "Pre-save/post-save text fidelity: PASS"
)


print(
    "=" * 110
)

NHANES 2009-2010 DIETARY DAY 1 + DAY 2 MERGE
DR1TOT_F shape: (10253, 166)
DR2TOT_F shape: (10253, 83)
PASS: SEQN exists in both files.
PASS: Both DR1TOT_F and DR2TOT_F contain 10,253 participant rows.

--- SEQN CHECK: DR1TOT_F ---
Rows: 10,253
Unique SEQN: 10,253
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN formatting: 0
PASS

--- SEQN CHECK: DR2TOT_F ---
Rows: 10,253
Unique SEQN: 10,253
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN formatting: 0
PASS

PASS: SEQN integrity validated in both files.

--- PARTICIPANT SET CHECK ---
Only in DR1TOT_F: 0
Only in DR2TOT_F: 0
PASS: Both files contain the same 10,253 participant SEQN.

--- SHARED VARIABLES DETECTED ---
['WTDRD1', 'WTDR2D', 'DRABF', 'DRDINT']
PASS: Exactly four expected shared variables found.

--- SHARED VARIABLE FIDELITY CHECK ---

WTDRD1
  Both populated: 9,754
  Missing-position mismatches: 0
  True numeric mismatches: 0
  Text-format differences: 0

WTDR2D
  Both populated: 8,406
  Miss